# 13. 강건성 점검 CV: 검증 fold에 가짜 변이를 넣으면 LB 순서를 맞히는가 (H13)

## 배경
- 이 대회는 CV 순위와 LB 순위가 자주 어긋남
  - 07 E: CV 0.4188(최고) → LB 0.2706(최저)
  - 10 vs 12-k20: CV 0.4113 vs 0.4115(동일) → LB 0.3206 vs 0.3051(−0.0155)
- 07에서 test의 변이 비율이 train보다 높아 보였음 → 변이가 늘어난 상황에서 버티는 모델이 LB에서 강할 가능성
- 리뷰 제안: 검증 fold에 가짜 승객(passenger) 변이를 추가해 다시 채점

## 가설
- H13: 가짜 변이를 넣은 CV는 일반 CV보다 LB 순서를 잘 맞힌다

## 검증 방법
- LB가 있는 5개 모델: 05 LR, 07 E LR, 09 XGB, 10 XGB, 12-k20 XGB
- 학습은 원래 데이터로, **검증 fold만** 변이를 추가해서 채점
- 추가할 변이 수: 환자 원래 변이 수 × 비율 (0 / 0.5 / 1 / 2)
  - 0 = 일반 CV → 기존 CV 점수 재현으로 코드 확인
  - 특정 비율에 맞추지 않으려고 여러 수준을 모두 비교
- 추가 위치: fold 학습 부분의 유전자별 변이 빈도에 비례해서 무작위 선택 (검증·test 정보 미사용)
- 모든 모델이 같은 가짜 변이 검증셋으로 채점 (fold·비율별 난수 고정)
- 판단: 비율별 점수와 LB의 Spearman 순위 상관이 일반 CV보다 높고, 07 E와 12-k20을 낮게 평가하면 채택
  → 남은 제출 후보를 고르는 보조 기준으로 사용

#### 준비: 데이터와 피처 세트
- 5개 모델이 쓰는 피처 세트 3종을 미리 만듦
  - any: 05 LR
  - any + stats: 09, 10, 12
  - any + trunc + stats: 07 E LR
- 모든 모델이 같은 fold로 평가받도록 `FOLDS`를 한 번만 만듦

In [1]:
import sys; sys.path.append('..')
import numpy as np, pandas as pd, xgboost as xgb
from scipy.stats import spearmanr
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_sample_weight
from src.config import SEED
from src.data import load_data, get_gene_cols
from src.features import mutation_features, MUT_TYPES
from src.validation import seed_everything, encode_target, get_folds, macro_f1

seed_everything()
train, test, submission = load_data()
genes = get_gene_cols(train)
y, le = encode_target(train)

F = mutation_features(train, genes)
ANY   = F['any'].astype('float32')
TRUNC = F['trunc'].astype('float32')
STATS = F['stats']
STAT_COLS = list(STATS.columns)
TYPE_LOG  = [f'n_{t}_log' for t in MUT_TYPES]      # 유형별 변이 수 (log1p)

X_any             = ANY
X_any_stats       = pd.concat([ANY, STATS], axis=1)
X_any_trunc_stats = pd.concat([ANY, TRUNC, STATS], axis=1)
FOLDS = get_folds(y)

#### LB가 있는 5개 모델 (각 노트북과 같은 설정)
- 07 E는 07 노트북처럼 stats만 표준화 (fold 안에서 fit)
- 12-k20은 12의 `RareGeneFilter`를 그대로 사용
- 각 모델 옆에 실제 LB 점수를 같이 기록

In [2]:
class BalancedXGB(xgb.XGBClassifier):
    def fit(self, X, y, **kw):
        return super().fit(X, y, sample_weight=compute_sample_weight('balanced', y), **kw)

class RareGeneFilter(BaseEstimator, TransformerMixin):
    def __init__(self, gene_cols, min_count=20):
        self.gene_cols, self.min_count = gene_cols, min_count
    def fit(self, X, y=None):
        n = X[self.gene_cols].sum()
        self.drop_ = n.index[n < self.min_count].tolist()
        return self
    def transform(self, X):
        return X.drop(columns=self.drop_)

BASE  = dict(tree_method='hist', eval_metric='mlogloss', random_state=SEED, n_jobs=-1)
XGB10 = dict(n_estimators=200, learning_rate=0.1, max_depth=3, colsample_bytree=0.3)
lr = lambda: LogisticRegression(C=0.1, max_iter=3000, class_weight='balanced', random_state=SEED)

MODELS = {   # 이름: (모델 만드는 함수, 피처, 실제 LB)
    '05_LR':      (lr, X_any, 0.2739),
    '07E_LR':     (lambda: make_pipeline(ColumnTransformer([('sc', StandardScaler(), STAT_COLS)],
                                                           remainder='passthrough'), lr()),
                   X_any_trunc_stats, 0.2706),
    '09_XGB':     (lambda: BalancedXGB(**BASE, n_estimators=100, learning_rate=0.1, max_depth=6),
                   X_any_stats, 0.2995),
    '10_XGB':     (lambda: BalancedXGB(**BASE, **XGB10), X_any_stats, 0.3206),
    '12_XGB_k20': (lambda: make_pipeline(RareGeneFilter(genes, 20), BalancedXGB(**BASE, **XGB10)),
                   X_any_stats, 0.3051),
}

#### 가짜 변이 검증셋 (fold × 비율별로 한 번만 만들어 모든 모델이 공유)
- 환자마다 추가할 변이 수 = 포아송(원래 변이 수 × 비율) → 변이 많은 환자에게 더 많이 추가
- 추가할 유전자는 fold 학습 부분의 유전자별 변이 빈도에 비례해서 고름 (자주 변이되는 유전자일수록 잘 걸림)
- 이미 변이가 있는 유전자에 또 걸리면 변화 없음 → 실제 추가된 수(`added`)를 따로 기록
- 난수는 fold·비율별로 고정 → 모든 모델이 똑같은 검증셋으로 채점받음

In [3]:
RATIOS = [0, 0.5, 1.0, 2.0]

def perturb_genes(G, p, ratio, rng):
    G = G.copy()
    n_pos = int((p > 0).sum())
    n0 = G.sum(1)
    for i, k in enumerate(rng.poisson(n0 * ratio)):
        if k:
            G[i, rng.choice(G.shape[1], size=min(k, n_pos), replace=False, p=p)] = 1
    return G, G.sum(1) - n0

PERTURB = {}
for f, (tr, va) in enumerate(FOLDS):
    p = ANY.iloc[tr].to_numpy().mean(0); p = p / p.sum()      # fold 학습 부분 빈도만 사용
    G0 = ANY.iloc[va].to_numpy()
    for r in RATIOS:
        rng = np.random.default_rng(SEED + 1000 * f + int(r * 10))
        PERTURB[(f, r)] = perturb_genes(G0, p, r, rng) if r else (G0.copy(), np.zeros(len(va)))

for r in RATIOS[1:]:   # 실제로 변이가 얼마나 늘었는지 확인
    before = ANY.to_numpy().sum()
    after = sum(PERTURB[(f, r)][0].sum() for f in range(len(FOLDS)))
    print(f'비율 {r}: 변이 유전자 수 {before:,.0f} → {after:,.0f} ({after / before:.2f}배)')

비율 0.5: 변이 유전자 수 218,893 → 314,761 (1.44배)
비율 1.0: 변이 유전자 수 218,893 → 412,507 (1.88배)
비율 2.0: 변이 유전자 수 218,893 → 608,893 (2.78배)


#### 피처 세트에 가짜 변이 반영
- 유전자 0/1 열을 가짜 변이가 들어간 행렬로 교체
- stats가 있으면 같이 갱신 (가짜 변이는 미스센스로 가정)
  - 유형별 개수를 log에서 되살리고 → 미스센스에 추가 수를 더하고 → 다시 log
  - n_gene_log, silent_ratio도 새 개수로 다시 계산
- trunc 열은 그대로 (미스센스는 절단 변이가 아님)

In [4]:
def apply_perturb(Xv, G, added):
    Xv = Xv.copy()
    Xv[genes] = G.astype(Xv[genes].dtypes.iloc[0])
    if 'n_gene_log' in Xv.columns:
        cnt = np.expm1(Xv[TYPE_LOG].to_numpy())
        cnt[:, TYPE_LOG.index('n_missense_log')] += added
        Xv[TYPE_LOG] = np.log1p(cnt)
        Xv['n_gene_log'] = np.log1p(G.sum(1))
        Xv['silent_ratio'] = cnt[:, TYPE_LOG.index('n_silent_log')] / np.maximum(cnt.sum(1), 1)
    return Xv

#### 5개 모델 × 4개 비율 채점
- fold마다 모델은 한 번만 학습하고, 비율별 검증셋으로 채점만 반복 → 시간 절약
- x0 열은 일반 CV와 같아야 함 (05 0.3818, 07E 0.4188, 09 0.3879, 10 0.4113, 12 0.4115)

In [5]:
rows = []
for name, (mk, X, lb) in MODELS.items():
    scores = {r: [] for r in RATIOS}
    for f, (tr, va) in enumerate(FOLDS):
        m = mk(); m.fit(X.iloc[tr], y[tr])
        for r in RATIOS:
            G, added = PERTURB[(f, r)]
            Xv = apply_perturb(X.iloc[va], G, added)
            scores[r].append(macro_f1(y[va], m.predict_proba(Xv).argmax(1)))
    row = {'model': name, 'LB': lb}
    for r in RATIOS:
        row[f'x{r}'] = round(float(np.mean(scores[r])), 4)
    rows.append(row); print(row)

res13 = pd.DataFrame(rows).sort_values('LB', ascending=False)
res13

{'model': '05_LR', 'LB': 0.2739, 'x0': 0.3818, 'x0.5': 0.3474, 'x1.0': 0.3166, 'x2.0': 0.2627}
{'model': '07E_LR', 'LB': 0.2706, 'x0': 0.4187, 'x0.5': 0.3677, 'x1.0': 0.3347, 'x2.0': 0.2751}
{'model': '09_XGB', 'LB': 0.2995, 'x0': 0.388, 'x0.5': 0.3226, 'x1.0': 0.2507, 'x2.0': 0.1839}
{'model': '10_XGB', 'LB': 0.3206, 'x0': 0.4116, 'x0.5': 0.353, 'x1.0': 0.288, 'x2.0': 0.2157}
{'model': '12_XGB_k20', 'LB': 0.3051, 'x0': 0.4116, 'x0.5': 0.3521, 'x1.0': 0.2831, 'x2.0': 0.2167}


,model,LB,x0,x0.5,x1.0,x2.0
3,10_XGB,0.3206,0.4116,0.3530,0.2880,0.2157
4,12_XGB_k20,0.3051,0.4116,0.3521,0.2831,0.2167
2,09_XGB,0.2995,0.3880,0.3226,0.2507,0.1839
0,05_LR,0.2739,0.3818,0.3474,0.3166,0.2627
1,07E_LR,0.2706,0.4187,0.3677,0.3347,0.2751


#### LB 순서를 얼마나 맞히는가
- Spearman 순위 상관: 1이면 LB 순서와 완전히 같음, 0이면 무관, 음수면 반대
- 핵심 시험 문제 두 개를 따로 확인
  - 07 E를 LR 중에서도 낮게 평가하는가 (일반 CV는 1등으로 평가)
  - 12-k20을 10보다 낮게 평가하는가 (일반 CV는 같게 평가)

In [6]:
for r in RATIOS:
    rho = spearmanr(res13[f'x{r}'], res13['LB']).correlation
    gap = res13.set_index('model').loc['10_XGB', f'x{r}'] - res13.set_index('model').loc['12_XGB_k20', f'x{r}']
    best = res13.loc[res13[f'x{r}'].idxmax(), 'model']
    label = '일반 CV' if r == 0 else f'가짜 변이 x{r}'
    print(f'{label:12s} | Spearman(LB) {rho:+.2f} | 10−12 차이 {gap:+.4f} | 1등 모델 {best}')

일반 CV        | Spearman(LB) -0.05 | 10−12 차이 +0.0000 | 1등 모델 07E_LR
가짜 변이 x0.5   | Spearman(LB) -0.10 | 10−12 차이 +0.0009 | 1등 모델 07E_LR
가짜 변이 x1.0   | Spearman(LB) -0.60 | 10−12 차이 +0.0049 | 1등 모델 07E_LR
가짜 변이 x2.0   | Spearman(LB) -0.70 | 10−12 차이 -0.0010 | 1등 모델 07E_LR


## 결과

| 모델 | LB | 일반 CV (x0) | x0.5 | x1.0 | x2.0 |
| :--- | :-: | :-: | :-: | :-: | :-: |
| 10 XGB | 0.3206 | 0.4116 | 0.3530 | 0.2880 | 0.2157 |
| 12 XGB k20 | 0.3051 | 0.4116 | 0.3521 | 0.2831 | 0.2167 |
| 09 XGB | 0.2995 | 0.3880 | 0.3226 | 0.2507 | 0.1839 |
| 05 LR | 0.2739 | 0.3818 | 0.3474 | 0.3166 | 0.2627 |
| 07 E LR | 0.2706 | 0.4187 | 0.3677 | 0.3347 | 0.2751 |

| 평가 방식 | Spearman(LB) | 10−12 차이 | 1등 모델 |
| :--- | :-: | :-: | :-: |
| 일반 CV | −0.05 | +0.0000 | 07 E LR |
| 가짜 변이 x0.5 | −0.10 | +0.0009 | 07 E LR |
| 가짜 변이 x1.0 | −0.60 | +0.0049 | 07 E LR |
| 가짜 변이 x2.0 | −0.70 | −0.0010 | 07 E LR |

## 결론
- **H13 기각**: 가짜 변이를 넣을수록 LB 순서와 더 반대가 됨 (Spearman −0.05 → −0.70)
- 무작위 변이에는 LR이 강하고(x2.0에서 0.26~0.28 유지) XGB가 약함(0.18~0.22)
  - LR: 수천 개 유전자의 작은 가중치 합 → 잡음이 평균에 묻힘
  - XGB: 소수의 분기에 크게 의존 → 그 자리에 잡음이 떨어지면 판단이 바뀜
- 그런데 실제 LB에서는 XGB가 이김 → **test와 train의 차이는 "무작위 변이 증가"가 아님**
  - 07의 "test 변이 비율이 높아 stats가 쏠림" 설명은 일부만 맞거나, 다른 종류의 차이가 더 큼
- 12-k20 판별도 실패 (10−12 차이 부호가 비율마다 바뀜)
- 이 점검은 제출 후보 선택 기준으로 쓰지 않음

## 다음
- LR과 XGB는 약점이 정반대 → 섞으면 서로의 실수를 보완할 가능성 → 14: LR + XGB 확률 섞기